# LFM2 — Basic Usage (MLX)

## Imports

In [1]:
from pprint import pprint

import mlx_lm

print("mlx-lm:", mlx_lm.__version__)

mlx-lm: 0.31.3


## Load Model and Tokenizer

In [3]:
MODEL_ID = "mlx-community/LFM2-700M-8bit"

model, tokenizer = mlx_lm.load(MODEL_ID)

print(f"Architecture: {model.model_type}")
print(f"Parameters: {mlx_lm.utils.get_total_parameters(model):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Architecture: lfm2
Parameters: 742,489,344


## Single Turn Generation

In [4]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [5]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.


## System Prompt

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|startoftext|><|im_start|>system
You are a helpful assistant who responds in all capitals.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



In [7]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)
print(response)

PARIS IS THE CAPITAL OF FRANCE.


## Multi-Turn Generation

In [8]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
)
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)

print(response)

The capital of France is Paris. It's a major city and a global center for art, fashion, gastronomy, and culture.


In [9]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant',
  'content': "The capital of France is Paris. It's a major city and a global center for art, fashion, gastronomy, and "
             'culture.'}]


In [10]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant',
  'content': "The capital of France is Paris. It's a major city and a global center for art, fashion, gastronomy, and "
             'culture.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [11]:
chat = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
)
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=128)

print(response)

One of the most famous landmarks in Paris is the Eiffel Tower. It's an iconic symbol of France and one of the most recognizable structures in the world. Completed in 1889 for the Exposition Universelle (World's Fair), it stands at 324 meters tall and offers breathtaking views of the city from its observation decks.


## Streaming Generation

In [12]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
)

for response in mlx_lm.stream_generate(model, tokenizer, chat, max_tokens=128):
    print(response.text, end="", flush=True)

The capital of France is Paris. It is a major city and global center for art, fashion, gastronomy, culture, and education. Paris is renowned for landmarks such as the Eiffel Tower, Louvre Museum, Notre-Dame Cathedral, and the historic Latin Quarter. It is also known for its influence on politics, fashion, and the arts worldwide.